# 📈 Notebook 1: Ghana GDP & Growth Trajectory

> **Objective:** How has Ghana's economy grown over 25 years — and what forces shaped that growth?

**Data source:** World Bank Open Data (`data/raw/gdp_indicators.csv`)

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sys, warnings
sys.path.append('../src')
from plot_utils import apply_style, annotate_events, save_fig, format_pct, PALETTE, GHANA_EVENTS

warnings.filterwarnings('ignore')
apply_style()
print('Setup complete ✅')

## 2. Load Data

In [ ]:
df_raw = pd.read_csv('../data/raw/gdp_indicators.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
# Separate Ghana and SSA
ghana = df_raw[df_raw['country'] == 'Ghana'].copy().sort_values('date').reset_index(drop=True)
ssa   = df_raw[df_raw['country'] != 'Ghana'].copy().sort_values('date').reset_index(drop=True)

print(f'Ghana records: {len(ghana)} | Years: {ghana["date"].min()} → {ghana["date"].max()}')
ghana.tail()

## 3. Chart: GDP Per Capita — Ghana vs Sub-Saharan Africa

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(ghana['date'], ghana['gdp_per_capita_usd'], color=PALETTE[0], linewidth=2.5, label='Ghana')
ax.plot(ssa['date'],   ssa['gdp_per_capita_usd'],   color=PALETTE[1], linewidth=2,   linestyle='--', label='Sub-Saharan Africa (avg)')
ax.fill_between(ghana['date'], ghana['gdp_per_capita_usd'], ssa['gdp_per_capita_usd'],
                where=ghana['gdp_per_capita_usd'] >= ssa['gdp_per_capita_usd'],
                alpha=0.08, color=PALETTE[0], label='Ghana above SSA avg')

# Annotate oil
ax.annotate('Oil production\nbegins', xy=(2011, ghana.loc[ghana['date']==2011, 'gdp_per_capita_usd'].values[0]),
            xytext=(2012.5, ghana['gdp_per_capita_usd'].max() * 0.75),
            fontsize=9, color='#374151',
            arrowprops=dict(arrowstyle='->', color='#9CA3AF'))

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('GDP Per Capita: Ghana vs Sub-Saharan Africa (2000–2023)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('GDP Per Capita (Current USD)')
ax.legend()
sns.despine()
plt.tight_layout()
save_fig(fig, '01_gdp_per_capita_comparison.png')
plt.show()

## 4. Chart: GDP Growth Rate — Volatility Story

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

colors = [PALETTE[0] if v >= 0 else PALETTE[1] for v in ghana['gdp_growth_pct'].fillna(0)]
ax.bar(ghana['date'], ghana['gdp_growth_pct'], color=colors, edgecolor='white', linewidth=0.5)
ax.plot(ssa['date'], ssa['gdp_growth_pct'], color='#374151', linewidth=1.5,
        linestyle='--', label='SSA average', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(ghana['gdp_growth_pct'].mean(), color=PALETTE[0], linewidth=1.2,
           linestyle=':', label=f'Ghana avg: {ghana["gdp_growth_pct"].mean():.1f}%', alpha=0.8)

format_pct(ax)
ax.set_title('Ghana Annual GDP Growth Rate (2000–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('GDP Growth (%)')
ax.legend()
sns.despine()
plt.tight_layout()
save_fig(fig, '01_gdp_growth_rate.png')
plt.show()

## 5. Chart: Economic Structure Shift (Agriculture → Services)

In [ ]:
sector_cols = ['agriculture_pct_gdp', 'industry_pct_gdp', 'services_pct_gdp']
sector_labels = ['Agriculture', 'Industry', 'Services']
sector_colors = [PALETTE[0], PALETTE[2], PALETTE[1]]

fig, ax = plt.subplots(figsize=(13, 5))
bottom = np.zeros(len(ghana))
for col, label, color in zip(sector_cols, sector_labels, sector_colors):
    vals = ghana[col].fillna(0).values
    ax.bar(ghana['date'], vals, bottom=bottom, label=label, color=color, alpha=0.85, edgecolor='white')
    bottom += vals

ax.set_title('Ghana Economic Structure: Sectoral Composition of GDP (2000–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% of GDP')
ax.legend(loc='upper left')
sns.despine()
plt.tight_layout()
save_fig(fig, '01_gdp_sector_composition.png')
plt.show()

## 6. Key Findings — GDP & Growth

- **Finding 1:** Ghana's GDP per capita has consistently outpaced the Sub-Saharan Africa average since [YEAR], but remains vulnerable to commodity price cycles
- **Finding 2:** The 2011 oil boom produced Ghana's highest single-year growth rate ([X]%) — but also embedded a structural commodity dependency that amplified subsequent downturns
- **Finding 3:** Services have grown as a share of GDP from ~[X]% to ~[X]% — signalling a gradual structural shift, though agriculture still employs the majority of the workforce
- **Finding 4:** The 2022 debt crisis produced the sharpest economic contraction since the data begins — a sharp reminder that headline growth numbers can mask fragility

---
*Continue to Notebook 2: Poverty & Inequality →*